In [ ]:
!pip -q install torch torchvision numpy tqdm

import torch, random, numpy as np

SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
random.seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)


Device: cuda


In [ ]:
from torchvision.datasets import STL10
import os

DATA_ROOT = "/content/data"
os.makedirs(DATA_ROOT, exist_ok=True)

_ = STL10(root=DATA_ROOT, split="unlabeled", download=True)
print("STL-10 ready")


100%|██████████| 2.64G/2.64G [04:33<00:00, 9.66MB/s]


STL-10 ready


In [ ]:
from torchvision import transforms
from PIL import ImageFilter
import random

class GaussianBlur:
    def __init__(self, sigma=(0.1, 2.0)):
        self.sigma = sigma
    def __call__(self, img):
        return img.filter(
            ImageFilter.GaussianBlur(radius=random.uniform(*self.sigma))
        )

class BYOLTransform:
    def __init__(self, size=96):
        self.t = transforms.Compose([
            transforms.RandomResizedCrop(size, scale=(0.2, 1.0)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomApply(
                [transforms.ColorJitter(0.4,0.4,0.4,0.1)], p=0.8
            ),
            transforms.RandomGrayscale(p=0.2),
            GaussianBlur(),
            transforms.ToTensor()
        ])
    def __call__(self, x):
        return self.t(x), self.t(x)


In [ ]:
from torch.utils.data import Dataset, DataLoader
from torchvision.datasets import STL10

class BYOLDataset(Dataset):
    def __init__(self, root):
        self.base = STL10(root=root, split="unlabeled", download=False)
        self.transform = BYOLTransform(96)

    def __len__(self):
        return len(self.base)  # 100,000

    def __getitem__(self, idx):
        img, _ = self.base[idx]
        return self.transform(img)

loader = DataLoader(
    BYOLDataset(DATA_ROOT),
    batch_size=256,      # SAME batch size you used earlier
    shuffle=True,
    drop_last=True,
    num_workers=2,
    pin_memory=True
)

print("Batches per epoch:", len(loader))  # ≈ 390


Batches per epoch: 390


In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import resnet18
import copy

class BYOL(nn.Module):
    def __init__(self, proj_dim=256, momentum=0.99):
        super().__init__()
        self.momentum = momentum

        # 🔴 MUST MATCH CHECKPOINT
        self.online_encoder = resnet18(weights=None)
        self.online_encoder.fc = nn.Identity()

        self.online_projector = nn.Sequential(
            nn.Linear(512, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Linear(1024, proj_dim)
        )

        self.predictor = nn.Sequential(
            nn.Linear(proj_dim, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Linear(1024, proj_dim)
        )

        # Target networks
        self.target_encoder = copy.deepcopy(self.online_encoder)
        self.target_projector = copy.deepcopy(self.online_projector)

        for p in self.target_encoder.parameters():
            p.requires_grad = False
        for p in self.target_projector.parameters():
            p.requires_grad = False

    @torch.no_grad()
    def update_target(self):
        for o, t in zip(self.online_encoder.parameters(),
                        self.target_encoder.parameters()):
            t.data = t.data * self.momentum + o.data * (1 - self.momentum)

        for o, t in zip(self.online_projector.parameters(),
                        self.target_projector.parameters()):
            t.data = t.data * self.momentum + o.data * (1 - self.momentum)

    def forward(self, x):
        h = self.online_encoder(x)
        z = self.online_projector(h)
        p = self.predictor(z)
        return p

    @torch.no_grad()
    def forward_target(self, x):
        h = self.target_encoder(x)
        z = self.target_projector(h)
        return z


In [ ]:
CKPT_PATH = "/content/drive/MyDrive/ssl_checkpoints/byol_epoch_167.pth"  # <-- CHANGE NAME IF NEEDED

model = BYOL().to(device)

state = torch.load(CKPT_PATH, map_location="cpu")
model.online_encoder.load_state_dict(state)

# VERY IMPORTANT: sync target encoder
model.target_encoder.load_state_dict(state)

print("Loaded BYOL checkpoint and synced target network")


Loaded BYOL checkpoint and synced target network


In [ ]:
def byol_loss(p, z):
    p = F.normalize(p, dim=1)
    z = F.normalize(z, dim=1)
    return 2 - 2 * (p * z).sum(dim=1).mean()

optimizer = torch.optim.Adam(
    list(model.online_encoder.parameters()) +
    list(model.online_projector.parameters()) +
    list(model.predictor.parameters()),
    lr=3e-4
)


In [ ]:
from torch.cuda.amp import autocast, GradScaler
from tqdm import tqdm

scaler = GradScaler()
EPOCHS = 20
SAVE_DIR = "/content/drive/MyDrive/ssl_checkpoints"
os.makedirs(SAVE_DIR, exist_ok=True)

print("Continuing BYOL training...")

for epoch in range(168, 168 + EPOCHS):
    model.train()
    total_loss = 0

    for (x1, x2) in tqdm(loader, desc=f"BYOL Epoch {epoch}"):
        x1, x2 = x1.to(device), x2.to(device)

        with autocast():
            p1 = model(x1)
            p2 = model(x2)

            with torch.no_grad():
                z1 = model.forward_target(x1)
                z2 = model.forward_target(x2)

            loss = byol_loss(p1, z2) + byol_loss(p2, z1)

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        model.update_target()
        total_loss += loss.item()

    avg = total_loss / len(loader)
    print(f"[Epoch {epoch}] Avg Loss: {avg:.4f}")

    torch.save(
        model.online_encoder.state_dict(),
        f"{SAVE_DIR}/byol_epoch_{epoch}.pth"
    )


/tmp/ipython-input-696737303.py:4: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


Continuing BYOL training...


BYOL Epoch 168:   0%|          | 0/390 [00:00<?, ?it/s]/tmp/ipython-input-696737303.py:18: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
BYOL Epoch 168: 100%|██████████| 390/390 [06:51<00:00,  1.05s/it]


[Epoch 168] Avg Loss: 0.5722


BYOL Epoch 169: 100%|██████████| 390/390 [06:48<00:00,  1.05s/it]


...

[Epoch 185] Avg Loss: 0.4684


BYOL Epoch 186: 100%|██████████| 390/390 [06:52<00:00,  1.06s/it]


[Epoch 186] Avg Loss: 0.4666


BYOL Epoch 187: 100%|██████████| 390/390 [07:03<00:00,  1.09s/it]


[Epoch 187] Avg Loss: 0.4671
